# Training and Evaluation Notebook

This notebook evaluates zero-shot news classification for Indian headlines using the Kaggle India Headlines dataset and includes an ablation study.

In [11]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import pipeline

## 1. Download Dataset

Option A: Download manually from Kaggle and place CSV in `data/`.

Option B: Use kagglehub in Python (requires Kaggle credentials configured).

In [12]:
# Uncomment this block if using kagglehub
#import kagglehub
#path = kagglehub.dataset_download('therohk/india-headlines-news-dataset')
#print('Dataset downloaded to:', path)

In [13]:
DATA_PATH = Path('../data/india_news_headlines.csv')
assert DATA_PATH.exists(), f'Missing dataset file at {DATA_PATH.resolve()}'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(3876557, 3)


,publish_date,headline_category,headline_text
0,20010102,unknown,Status quo will not be disturbed at Ayodhya; s...
1,20010102,unknown,Fissures in Hurriyat over Pak visit
2,20010102,unknown,America's unwanted heading for India?
3,20010102,unknown,For bigwigs; it is destination Goa
4,20010102,unknown,Extra buses to clear tourist traffic


## 2. Prepare Labels

Adjust the source column names to match your CSV. Expected text column: `headline_text`.
Expected label column: `category`.

In [14]:
# Auto-detect common column names across dataset versions
text_candidates = ['headline_text', 'title', 'headline', 'news_title']
label_candidates = ['category', 'headline_category', 'section', 'topic']

TEXT_COL = next((c for c in text_candidates if c in df.columns), None)
LABEL_COL = next((c for c in label_candidates if c in df.columns), None)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        'Could not detect required columns. '
        f'Found columns: {list(df.columns)}. '
        f'Tried text columns: {text_candidates}, label columns: {label_candidates}'
    )

print('Using TEXT_COL =', TEXT_COL, '| LABEL_COL =', LABEL_COL)

work = df[[TEXT_COL, LABEL_COL]].dropna().copy()
work[TEXT_COL] = work[TEXT_COL].astype(str).str.strip()
work[LABEL_COL] = work[LABEL_COL].astype(str).str.strip()
work = work[work[TEXT_COL] != '']

def map_to_app_label(raw_label: str):
    s = str(raw_label).lower()
    # Direct exact mapping first
    exact = {
        'politics': 'Politics',
        'sports': 'Sports',
        'technology': 'Technology',
        'business': 'Business',
        'entertainment': 'Entertainment',
    }
    if s in exact:
        return exact[s]

    # Keyword fallback for dataset variants (e.g., 'tech', 'bollywood', 'economy')
    if any(k in s for k in ['politic', 'election', 'government', 'parliament', 'policy']):
        return 'Politics'
    if any(k in s for k in ['sport', 'cricket', 'football', 'tennis', 'olympic']):
        return 'Sports'
    if any(k in s for k in ['tech', 'technology', 'science', 'gadget', 'digital', 'ai']):
        return 'Technology'
    if any(k in s for k in ['business', 'economy', 'finance', 'market', 'stock', 'startup']):
        return 'Business'
    if any(k in s for k in ['entertainment', 'bollywood', 'movie', 'film', 'cinema', 'tv', 'celebrity']):
        return 'Entertainment'

    return None

work['y'] = work[LABEL_COL].apply(map_to_app_label)
work = work.dropna(subset=['y'])
work['x'] = work[TEXT_COL]

print(work.shape)
work['y'].value_counts()

Using TEXT_COL = headline_text | LABEL_COL = headline_category
(1087758, 4)


y
Technology       721855
Business         165764
Sports           137816
Entertainment     41947
Politics          20376
Name: count, dtype: int64

In [15]:
train_df, test_df = train_test_split(
    work[['x', 'y']],
    test_size=0.2,
    random_state=42,
    stratify=work['y']
)

print('Train:', train_df.shape, 'Test:', test_df.shape)

Train: (870206, 2) Test: (217552, 2)


## 3. Zero-shot Evaluation

In [16]:
candidate_labels = ['Politics', 'Sports', 'Technology', 'Business', 'Entertainment']
clf = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

sample_test = test_df.sample(n=min(500, len(test_df)), random_state=42).reset_index(drop=True)
texts = sample_test['x'].tolist()
pred_obj = clf(
    texts,
    candidate_labels=candidate_labels,
    multi_label=False,
#    hypothesis_template='This news article is about {}.'
    hypothesis_template = 'This headline is about {}.'
)

pred_labels = [obj['labels'][0] for obj in pred_obj]
true_labels = sample_test['y'].tolist()

print(classification_report(true_labels, pred_labels, digits=4))

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

               precision    recall  f1-score   support

     Business     0.4014    0.7600    0.5253        75
Entertainment     0.1318    0.8500    0.2282        20
     Politics     0.0909    1.0000    0.1667         8
       Sports     0.7162    0.7361    0.7260        72
   Technology     0.8358    0.1723    0.2857       325

     accuracy                         0.3820       500
    macro avg     0.4352    0.7037    0.3864       500
 weighted avg     0.7134    0.3820    0.3809       500



In [17]:
cm = confusion_matrix(true_labels, pred_labels, labels=candidate_labels)
cm_df = pd.DataFrame(cm, index=candidate_labels, columns=candidate_labels)
cm_df

,Politics,Sports,Technology,Business,Entertainment
Politics,8,0,0,0,0
Sports,7,53,0,8,4
Technology,66,21,56,76,106
Business,5,0,11,57,2
Entertainment,2,0,0,1,17


## 4. Ablation Study

Compare two hypothesis templates to study prompt sensitivity in zero-shot classification.

In [19]:
templates = [
    'This news article is about {}.',
    'The primary topic of this Indian headline is {}.',
    'This headline is about {}.',
]

ablation_rows = []
for template in templates:
    preds = clf(
        texts,
        candidate_labels=candidate_labels,
        multi_label=False,
        hypothesis_template=template
    )
    yhat = [obj['labels'][0] for obj in preds]
    acc = (pd.Series(yhat) == pd.Series(true_labels)).mean()
    ablation_rows.append({'template': template, 'accuracy': acc})

pd.DataFrame(ablation_rows).sort_values('accuracy', ascending=False)

,template,accuracy
1,The primary topic of this Indian headline is {}.,0.410
0,This news article is about {}.,0.392
2,This headline is about {}.,0.382


## 5. Notes

- This notebook evaluates classification quality only.
- Summarization quality is evaluated manually in the technical report.
- Increase sample size and run-time if you need tighter confidence intervals.